<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/leaderboard_focus/GEMS_M1_CHAMPION_TO_SUBMISSION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS Milestone 1: champion → validated submission (canonical notebook, code rev m1.3-r1)

**One notebook, one configuration.** Checks the Drive artifacts read-only, recovers the strongest valid frozen W2 champion, reproduces the official evaluation, diagnoses export parity, applies explicit promotion gates, writes a validated GeoTIFF only when every mandatory gate passes (or, opt-in, a separately named exploratory baseline), and records a manually submitted public score. Nothing is trained, re-inferred or uploaded.

This notebook **replaces** `00_ENVIRONMENT_AND_ARTIFACTS` … `04_LEADERBOARD_REVIEW` (kept for history, superseded).

| | |
|---|---|
| Runtime | CPU, **High-RAM** (no GPU needed) |
| Inputs (read-only) | `MyDrive/GEMS_2/outputs/p6_fast_track/{stack, boost/benchmark, recovery/export_parity}`, `experiments/b0_reference__d54e938a`, `experiments/b1_bg_weight__7755635b`, folds, template, M9 selection |
| Outputs | `MyDrive/GEMS_2/outputs/leaderboard_focus/m1_champion_recovery/` |
| Code | `gems_bench/leaderboard_focus/workflow.py` (`Runner`) |

| stage | cell | writes | status meaning |
|---|---|---|---|
| 0 | configuration, bootstrap | `environment/` | – |
| 0c | Drive preflight (read-only, no engine) | `diagnostics/`, `manifests/preflight.json` | BLOCKED = a required input is unusable: engine stages refused |
| 1 | artifact inventory | `artifact_inventory.*`, `staging_manifest.json` | COMPLETE = every required artifact OK |
| 2 | frozen champion selection | `selection/` | COMPLETE = **selection** complete (not promotion) |
| 3 | official validation + budget / mask forensics | `validation/`, `diagnostics/`, `figures/` | COMPLETE / WARNING (caveats) / BLOCKED |
| 4 | export parity + promotion gates | `parity/`, `selection/promotion_gates.*` | technical validity and internal promotion, separately |
| 5 | submission export (champion) | `submissions/` | READY_FOR_UPLOAD only when every check passed |
| 5b | exploratory baseline (opt-in) | `submissions/exploratory_*` | EXPLORATORY_READY: technically valid, fails the geo gate, never the champion |
| 6 | leaderboard review | `leaderboard/` | manual entries only |
| 7 | final report | `reports/M1_*.md` | executive summary, failure diagnostics, next action, M1.3 repair report |

**Resume after a disconnect:** run Stage 0 again, then the stage cells. Finished stages print `SKIP_CACHED`; a BLOCKED stage is always re-evaluated; a stage whose code changed re-runs (its `impl` hash differs).

## Stage 0 · Configuration (the only cell you edit)

`mode`: `FULL` (every stage, stops downstream on a blocker) · `RECOVER` (inventory + selection) · `VALIDATE` (validation + parity on the persisted selection) · `SUBMIT` (export the persisted validated recipe if ready) · `PREFLIGHT` (read-only Drive check + report, no engine). Every mode starts with the preflight. A stage outside the mode prints `NOT_REQUIRED` and leaves its persisted result untouched.

`M1_CONFIG` overrides `configs/leaderboard_focus/m1.yaml` (normally empty). The geographic floor (0.1758) is **mandatory** by default: a champion below it is not exported as the champion. Two explicit, recorded choices exist: the opt-in **exploratory baseline** (recommended: separate file, status `EXPLORATORY_READY`, champion stays blocked) `M1_CONFIG = {'exploratory': {'enabled': True, 'reason': '<why>'}}`, or the older `promotion.geographic_gate: 'advisory'` + `override_note` (exports under the champion name with a WARNING).

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                       # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'FULL',                      # FULL | RECOVER | VALIDATE | SUBMIT | PREFLIGHT (read-only Drive check only)
    'reuse_cached': True,                # reuse staged files while the inventory fingerprint is unchanged
    'overwrite_existing': False,
    'strict_provenance': True,           # a changed B0 / B1 experiment id blocks M1

    'required_grids': ['boost_a'],       # the only grids read; unusable -> selection BLOCKED (no fallback)
    'train_cnn': False,                  # M1 never trains: True is refused
    'train_transformer': False,
    'run_extra_seeds': False,

    'save_figures': True,                # PNG files (their _data.csv is always written)
    'save_csv': True,                    # required (CSV tables are the stage contract)
    'save_parquet': True,
    'save_geotiff': True,                # False: stage 5 is NOT_REQUIRED
    'sync_to_drive': True,
}
PRIVATE_REPO = False
M1_CONFIG = {}                          # overrides of configs/leaderboard_focus/m1.yaml
# Opt-in exploratory baseline (default OFF; never the champion; a written reason is mandatory):
# M1_CONFIG = {'exploratory': {'enabled': True, 'reason': '<why a geo-gate-failing baseline is wanted>'}}
EXPECT = {'code_rev': 'm1.3-r1', 'commit': None}   # set 'commit' to the pushed SHA
LEADERBOARD_ENTRY = {                   # stage 6: fill ONLY after a manual DrivenData upload
    'submission_id': '', 'submission_file': '', 'public_dti': None, 'submitted_at': '', 'notes': '',
}

## Stage 0 · Bootstrap: Drive, strict clone, version lock, runner

A ref that cannot be resolved stops here (no fallback branch). The runtime must be fresh (`gems_bench` not yet imported). The P6 engine is attached lazily, once, by the first stage that needs it.

In [2]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)       # a SHA / tag must resolve, else this raises
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from Stage 0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'], f'code rev {LF.M1_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'

import pandas as pd
from IPython.display import Markdown, display
from gems_bench.leaderboard_focus.workflow import Runner
R = Runner(CONFIG, HEAD, M1_CONFIG)
print('Requested', ref, '| SHA', HEAD, '| M1 code rev', LF.M1_CODE_REV, '| mode', R.nb['mode'], '| stages', R.stages)
print('Geographic gate:', R.cfg['promotion']['geographic_gate'], '| floor', R.cfg['selection']['geo_floor'],
      '| grids', R.cfg['champion']['grids'], '| outputs', R.cfg['outputs'])

Mounted at /content/drive
Requested main | SHA 90a99ec1435253293854ee20d7027607c4486abb | M1 code rev m1.2-r1 | mode FULL | stages ('artifacts', 'selection', 'validation', 'parity', 'promotion', 'submissions', 'leaderboard', 'report')
Geographic gate: mandatory | floor 0.1758 | grids ['boost_a'] | outputs {'save_figures': True, 'save_parquet': True, 'save_geotiff': True, 'sync_to_drive': True}


## Stage 0c · Drive preflight (read-only, no engine)

Reads `MyDrive/GEMS_2` directly and writes only `outputs/leaderboard_focus/m1_champion_recovery/diagnostics/` (+ `manifests/preflight.json`). Per file: path, existence, size, mtime, readability, schema (CSV header / rows / duplicates / nulls, JSON / YAML keys, Parquet schema, NPY shape + sampled values, GeoTIFF grid / CRS / nodata + sampled coverage). Cross-checks: recipe ids, B0 / B1 experiment ids, template vs CNN raster alignment, champion cell present in the boost-A grid, stage code vs recorded `impl`. Every finding is labelled **OBSERVED / INFERRED / UNVERIFIED**.

It also re-reads the last run's `validation/zone_budget.csv` + `per_zone_metrics.csv` under the upper-bound budget contract (`diagnostics/budget_root_cause_preflight.csv`, `figures/per_zone_area_budget_preflight.png`).

`BLOCKED` = a required frozen input is missing / unreadable / malformed / misaligned: every engine stage then records `INCOMPLETE` without attaching the engine. Run it alone with `mode: PREFLIGHT`.

In [ ]:
PF = R.step('preflight')
D = R.light().ws
print('PREFLIGHT:', PF.get('status'), '|', PF.get('counts'))
for r in PF.get('reasons') or []:
    print('  blocking:', r)
for w in PF.get('warnings') or []:
    print('  warning:', w)
inv = D.load_table('diagnostics/drive_inventory.csv')
if len(inv):
    display(inv[inv.status != 'OK'][['scope', 'id', 'rel', 'required', 'status', 'detail']])   # non-OK files only
display(D.load_table('diagnostics/schema_checks.csv'))
display(D.load_table('diagnostics/artifact_compatibility.csv'))
B = D.load_table('diagnostics/budget_root_cause_preflight.csv')
if len(B):
    display(B[[c for c in ('zone', 'eligible_px', 'pool_px', 'positive_px', 'pool_yield', 'frac_of_eligible',
                           'cap_frac', 'eval_positive_px', 'positive_on_known_px', 'm1_1_two_sided_ok', 'ok',
                           'cause') if c in B]])

## Stage 1 · Artifact inventory and selective staging

Every artifact in `m1.yaml → artifacts` is checked on Drive: existence, sha256, experiment identity (B0 `b0_reference__d54e938a`, B1 `b1_bg_weight__7755635b`), required file names (`oof.tif`, `disc0-2.tif`; W2 `geo_0-2.npy`, `disc_official_0-2.npy`). Extra files are accepted and listed. Only `staged: true` files are copied locally — never boost B/C/D, W1 seeds, checkpoints or historical submission rasters.

Statuses: `OK`, `MISSING`, `MISSING_REQUIRED_FILES`, `COUNT_MISMATCH`, `AMBIGUOUS`, `ID_CHANGED`. A required artifact that is not OK stops every later stage (they record INCOMPLETE).

In [3]:
A = R.step('artifacts')
if 'inventory' in A:
    display(A['inventory'][['id', 'required', 'status', 'n_files', 'experiment_ids', 'extra_files', 'note']])

[sync in] 19 copied (263.2 MiB), 0 unchanged, 1s: ['outputs/m9_geofault_production/baseline/baseline_selection.json', 'outputs/p6_fast_track/boost/benchmark/grid.csv', 'outputs/p6_fast_track/boost/benchmark/manifest.json', 'outputs/p6_fast_track/boost/benchmark/status.json', 'outputs/p6_fast_track/recovery/export_parity/decision.json', 'outputs/p6_fast_track/recovery/export_parity/policy_comparison.csv', 'outputs/p6_fast_track/recovery/export_parity/train_inference_parity.csv', 'outputs/p6_fast_track/recovery/export_parity/unit_overlap.csv', 'outputs/p6_fast_track/stack/candidate.json', 'outputs/p6_fast_track/stack/features/rows.parquet', 'outputs/p6_fast_track/stack/grid.csv', 'outputs/p6_fast_track/stack/maps/disc_official_0.npy', 'outputs/p6_fast_track/stack/maps/disc_official_1.npy', 'outputs/p6_fast_track/stack/maps/disc_official_2.npy', 'outputs/p6_fast_track/stack/maps/full_score.tif', 'outputs/p6_fast_track/stack/maps/geo_0.npy', 'outputs/p6_fast_track/stack/maps/geo_1.npy', 'o

,id,required,status,n_files,experiment_ids,extra_files,note
0,w2_candidate,True,OK,1,,[],
1,w2_rows,True,OK,1,,[],
2,w2_stack_grid,True,OK,1,,[],
3,w2_maps,True,OK,6,,[],
4,w2_full_score_hist,False,OK,1,,[],
5,boost_a_grid,True,OK,1,,[],
6,boost_a_manifest,True,OK,1,,[],
7,boost_a_status,False,OK,1,,[],
8,recovery_decision,False,OK,1,,[],
9,recovery_parity,False,OK,1,,[],


## Stage 2 · Frozen champion selection

Loads **only** `required_grids` (boost A) through `full_benchmark.load_grids(fg, sources=…, required=…)`; an absent, empty, header-only or malformed required grid makes the stage BLOCKED with the file facts in `selection/grid_provenance.csv`. Selection = the predeclared p6g rule (nested single policy, objective disc, worst zone ≥ 0.153414). COMPLETE here means *selection complete*: promotion is decided in Stage 4.

In [4]:
S2 = R.step('selection')
W = R.light().ws
display(W.load_table('selection/grid_provenance.csv'))
C = W.load_table('selection/champion_candidates.csv')
if len(C):
    display(C[[c for c in ('candidate_id', 'role', 'status', 'eligible', 'disc_dti', 'disc_worst', 'geo_dti',
                           'geo_floor_met', 'decision') if c in C]])

[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/config_snapshot.json', 'reports/pipeline/environment_report.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/bootstrap.json']
[bootstrap] DONE: bootstrap: ok (3 checks)
[bootstrap] outputs: ['/content/drive/MyDrive/GEMS_2/environment/pipeline/config_snapshot.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/environment_report.csv']
[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['data/raw/acquisition_manifest.json', 'reports/pipeline/raw_data_acquisition.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/raw_data.json']
[raw_data] DONE: raw_data: ok (3 checks)
[raw_data] outputs: ['/content/drive/MyDrive/GEMS_2/data/raw/acquisition_manifest.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/raw_data_acquisition.csv']
[sync in] 1 copied (0.0 MiB), 0 unchanged, 1s: ['environment/pipeline/native_grid.json']
[native_grid] SKIP_CACHED: native_g

,source,required,usable,grid_status,detail,path,bytes,rows,sha256,stage_status
0,boost_a,True,True,OK,NaN,boost/benchmark/grid.csv,99538,300,c70ca4c62b85742da8236040186edb31d66cd58e6b15e7...,COMPLETE


,candidate_id,role,status,eligible,disc_dti,disc_worst,geo_dti,geo_floor_met,decision
0,ref_p6_w2,reference,EVALUATED,True,0.224640,0.153414,0.102664,False,BASELINE
1,a_w2_crest,candidate,EVALUATED,True,0.238882,0.192575,0.114259,False,ELIGIBLE_NOT_CHOSEN
2,a_w2_skeleton,candidate,EVALUATED,True,0.242156,0.198579,0.105687,False,ELIGIBLE_NOT_CHOSEN
3,a_w2_rep_frac,candidate,EVALUATED,True,0.242156,0.198579,0.105687,False,CHAMPION
4,a_hybrid,candidate,EVALUATED,False,0.228417,0.153414,0.126658,False,INELIGIBLE
5,rec_p6r_choice,recovery_baseline,EVALUATED,True,0.242156,0.198579,0.105687,False,ELIGIBLE_NOT_CHOSEN
6,ref_m9_fallback,reference,EVALUATED,False,0.162482,0.126160,0.158416,False,INELIGIBLE
7,m1_w2_uploaded_cell,historical,EVALUATED,False,0.178905,0.155761,0.168841,False,INELIGIBLE


## Stage 3 · Official validation

Per held-out zone the champion cell is rebuilt from the frozen maps and scored by the existing official scorer; components must equal the saved grid (1e-6). Recorded historical values are reproduced from the same grid and labelled by protocol: a `separate`-protocol geographic value (geo cell chosen on geo labels) is **not comparable** with the champion's single-policy geo DTI, so its gap is explained, not treated as a defect (`validation/protocol_comparison.csv`). The dry zone-stitched build checks template, coverage, per-zone budget and recipe.

**M1.3 budget contract** (the one the written file is audited with): `frac` is the share of the eligible domain the selector takes; a thin representation keeps part of it, so a zone fails only on area > frac × (1 + tol), a positive pixel on a mapped fault, or an empty map. The superseded two-sided verdict is kept as `m1_1_two_sided_ok`. New diagnostics: `diagnostics/representation_comparison.csv` (evaluated vs deployed binary map per zone, disagreement split into *official mask* vs *re-ranked pool*), `mask_alignment.csv`, `root_cause_summary.json`, `evaluated_vs_exported_maps.npz`, and the figures `per_zone_area_budget`, `eligible_mask_comparison`, `evaluated_vs_exported`, `geographic_score_comparison` (each with `_data.csv`).

In [ ]:
S3 = R.step('validation')
W = R.light().ws
display(W.load_table('validation/official_scores.csv'))
Z = W.load_table('validation/zone_budget.csv')
if len(Z):
    display(Z[[c for c in ('zone', 'eligible_px', 'eval_eligible_px', 'known_in_zone_px', 'pool_px', 'positive_px',
                           'pool_yield', 'frac_of_eligible', 'cap_frac', 'eval_positive_px', 'deploy_vs_eval_px',
                           'positive_on_known_px', 'm1_1_two_sided_ok', 'ok', 'cause') if c in Z]])
display(W.load_table('diagnostics/representation_comparison.csv'))
print('root cause:', (W.read_json('diagnostics/root_cause_summary.json') or {}).get('blocking_classes'),
      '| diagnostics:', (W.read_json('manifests/diagnostics.json') or {}).get('status'))
display(W.load_table('validation/protocol_comparison.csv'))

## Stage 4 · Export parity, then promotion gates

Parity: fold assignment, duplicated / missing pixels, zone score vs evaluated score (Spearman ≥ 0.999, strict), recipe identity; historical-export rows are documented and never block the champion. Then the promotion gates, each PASS / WARNING / BLOCKED / PENDING: `SELECTION_COMPLETE`, `DISCOVERY_ELIGIBLE`, `GEOGRAPHIC_GATE_PASS`, `VALIDATION_PASS`, `PARITY_PASS` → `READY_FOR_EXPORT`.

In [ ]:
S4 = R.step('parity')
W = R.light().ws
P = W.load_table('parity/parity_report.csv')
if len(P):
    display(P[P.scope == 'champion'][['check', 'unit', 'status', 'detail']])
G = R.step('promotion')
display(W.load_table('selection/promotion_gates.csv'))
PG = W.read_json('selection/promotion_gates.json') or {}
print('TECHNICAL VALIDITY:', PG.get('technical_validity'), '| INTERNAL PROMOTION:', PG.get('internal_promotion'),
      '| exploratory eligible:', PG.get('exploratory_eligible'))

## Stage 5 · Submission export

Runs only when promotion is READY_FOR_EXPORT; otherwise it records BLOCKED / INCOMPLETE and writes **no** file. The written band must equal the array validated in Stage 3 (sha256), the recipe must equal the Stage-2 recipe, and the GeoTIFF contract (1 band, float32, EPSG 32611, 100 m, values in [0, 1], ≤ 50 MB) must hold. Versioned name `submission_m1_<family>_<recipe_id>.tif`; never overwritten; never uploaded.

In [7]:
S5 = R.step('submissions')
sub = R.light().ws.read_json('submissions/submission_manifest.json') or {}
print('SUBMISSION:', sub.get('status'), '|', sub.get('file'), '| sha256', sub.get('sha256'))
for r in sub.get('reasons') or []:
    print('  reason:', r)

[sync out] 3 copied (0.0 MiB), 48 unchanged, 0s: ['outputs/leaderboard_focus/m1_champion_recovery']
[m1] submissions  BLOCKED | promotion BLOCKED (READY_FOR_EXPORT not reached): see selection/promotion_gates.csv
SUBMISSION: BLOCKED | None | sha256 None
  reason: promotion BLOCKED (READY_FOR_EXPORT not reached): see selection/promotion_gates.csv


## Stage 5b · Exploratory baseline (opt-in, default OFF)

`NOT_REQUIRED` (no engine attach) unless `M1_CONFIG = {'exploratory': {'enabled': True, 'reason': '<why>'}}`. Runs only when **technical validity** passes (selection, validation, parity PASS / WARNING), discovery eligibility passes, and `GEOGRAPHIC_GATE_PASS` is the **only** failing gate. Same writer, same official-format / eligible-mask / parity / sha256 checks as the champion; file `submissions/exploratory_m1_<family>_<recipe_id>.tif`, manifest `submissions/exploratory_manifest.json`, `exploratory_sha256.txt`, `EXPLORATORY_README.md`. Best status `EXPLORATORY_READY`, never `READY_FOR_UPLOAD`; the champion submission stays `BLOCKED`; nothing is uploaded.

In [ ]:
S5b = R.step('exploratory')
ex = R.light().ws.read_json('submissions/exploratory_manifest.json') or {}
print('EXPLORATORY:', S5b.get('status'), '|', ex.get('status'), '|', ex.get('file'), '| sha256', ex.get('sha256'),
      '| validated champion:', ex.get('validated_champion', False))
for r in S5b.get('reasons') or []:
    print('  reason:', r)

## Stage 6 · Leaderboard review (manual)

1. Only when Stage 5 says `READY_FOR_UPLOAD`: download the file from Drive, check `sha256sum` against `submissions/submission_sha256.txt`, upload it by hand on DrivenData (competition 306; 3 scored submissions per rolling 7 days).
2. Fill `LEADERBOARD_ENTRY` in Stage 0 (`submitted_at` as `YYYY-MM-DD`) and run this cell. An entry for another file, a changed file or a duplicate id is refused.

Public scores are recorded for the history only: they never feed back into selection.

In [8]:
S6 = R.step('leaderboard', entry=LEADERBOARD_ENTRY)
display(R.light().ws.load_table('leaderboard/leaderboard_history.csv'))

[sync out] 3 copied (0.0 MiB), 51 unchanged, 0s: ['outputs/leaderboard_focus/m1_champion_recovery']
[m1] leaderboard  NOT_REQUIRED


""


## Stage 7 · Final report

Written from the persisted manifests: `reports/M1_EXECUTIVE_SUMMARY.md`, `reports/M1_FAILURE_DIAGNOSTICS.md` (every non-passing stage, matched against the known failure modes), `reports/M1_NEXT_ACTION.md`, `reports/stage_status.csv`.

In [9]:
REP = R.step('report')
display(Markdown(R.light().ws.path('reports/M1_EXECUTIVE_SUMMARY.md').read_text(encoding='utf-8')))
display(R.table())

[sync out] 4 copied (0.0 MiB), 54 unchanged, 0s: ['outputs/leaderboard_focus/m1_champion_recovery']
[m1] report       COMPLETE


# M1 executive summary

Offline numbers are nested held-out estimates from the frozen boost-A grid, NOT leaderboard scores.

| item | value |
|---|---|
| Selected candidate | a_w2_rep_frac |
| Recipe id | 0a8692b2ab |
| Discovery DTI (nested, single policy) | 0.24215604329651 |
| Worst discovery zone DTI | 0.19857922135752035 |
| Geographic DTI (nested, single policy) | 0.10568748897234223 |
| Geographic floor | 0.1758 -> BLOCKED (mandatory) |
| Validation | BLOCKED |
| Parity | INCOMPLETE |
| Promotion (READY_FOR_EXPORT) | BLOCKED |
| Submission status | BLOCKED |
| GeoTIFF (Drive) | none |

## Blocking reasons

* validation: per-zone area budget / known-fault mask violated on zones [0, 1, 2]
* parity: upstream stages not usable: {'validation': 'BLOCKED'}
* promotion: GEOGRAPHIC_GATE_PASS: geo_dti 0.1056874889723422 < floor 0.1758 (nested single policy). 0 of 8 evaluated candidates reach the floor
* promotion: VALIDATION_PASS: stage BLOCKED: per-zone area budget / known-fault mask violated on zones [0, 1, 2]
* promotion: PARITY_PASS: stage INCOMPLETE: upstream stages not usable: {'validation': 'BLOCKED'}
* submissions: promotion BLOCKED (READY_FOR_EXPORT not reached): see selection/promotion_gates.csv

## Recommended next action

Stage `validation` is BLOCKED: fix the reasons in reports/M1_FAILURE_DIAGNOSTICS.md, then re-run with mode FULL (finished stages are reused).


,stage,in_mode,status,written,first_reason
0,artifacts,True,COMPLETE,2026-10-28T21:59:48,
1,selection,True,COMPLETE,2026-10-28T22:02:43,
2,validation,True,BLOCKED,2026-10-28T22:03:17,per-zone area budget / known-fault mask violat...
3,parity,True,INCOMPLETE,2026-10-28T22:03:18,upstream stages not usable: {'validation': 'BL...
4,promotion,True,BLOCKED,2026-10-28T22:03:18,GEOGRAPHIC_GATE_PASS: geo_dti 0.10568748897234...
5,submissions,True,BLOCKED,2026-10-28T22:03:18,promotion BLOCKED (READY_FOR_EXPORT not reache...
6,leaderboard,True,NOT_REQUIRED,2026-10-28T22:03:18,


## Verification checklist

0. Stage 0c preflight `COMPLETE` (or `WARNING`: read each); `diagnostics/DRIVE_READINESS_REPORT.md` lists no required frozen input as non-OK; `budget_root_cause_preflight.csv` shows the last run's zones with `positive_on_known_px = 0`.
1. Stage 1 `COMPLETE`; `b0_maps` / `b1_maps` show the recorded experiment ids and `oof_fold0-2.tif` as extras.
2. Stage 2 `COMPLETE`; `grid_provenance.csv` shows `boost_a` usable and nothing else loaded.
3. Stage 3 `COMPLETE` or `WARNING` (read every warning); `zone_budget.csv → ok` True on every zone with `cause` explaining it; `representation_comparison.csv` shows `deploy_outside_eligible_px = 0`; `protocol_comparison.csv` explains each recorded value.
4. Stage 4 parity `COMPLETE`; promotion: read **technical validity** and **internal promotion** separately.
5. Stage 5 `READY_FOR_UPLOAD` before any champion upload; `export_parity_report.json → equal: true`. If only the geographic gate fails and you opted in: Stage 5b `EXPLORATORY_READY` (never the champion).
6. `reports/M1_NEXT_ACTION.md` names the next step; `reports/M1_3_REPAIR_REPORT.md` quotes the budget repair.

## Troubleshooting

See `reports/M1_FAILURE_DIAGNOSTICS.md` (generated) and `docs/leaderboard_focus/TROUBLESHOOTING.md`. Common cases: `preflight BLOCKED` → restore the file named in `DRIVE_READINESS_REPORT.md`; `required grid(s) unusable` → restore `outputs/p6_fast_track/boost/benchmark/grid.csv`; `produced by different stage code` → run mode `FULL`; `GEOGRAPHIC_GATE_PASS BLOCKED` → a decision (keep blocked, or the exploratory baseline); `gems_bench already imported` → restart the session.

Milestone 1 is complete only after a `READY_FOR_UPLOAD` file exists and you have verified it. An exploratory upload gives a leaderboard baseline, not a validated champion. Milestone 2 is not started here.